In [1]:
%pip uninstall --yes 'keras' 'matplotlib' 'scikit-learn' 'tensorflow'

Found existing installation: keras 3.10.0
Uninstalling keras-3.10.0:
  Successfully uninstalled keras-3.10.0
Found existing installation: matplotlib 3.10.0
Uninstalling matplotlib-3.10.0:
  Successfully uninstalled matplotlib-3.10.0
Found existing installation: scikit-learn 1.6.1
Uninstalling scikit-learn-1.6.1:
  Successfully uninstalled scikit-learn-1.6.1
Found existing installation: tensorflow 2.19.0
Uninstalling tensorflow-2.19.0:
  Successfully uninstalled tensorflow-2.19.0
Note: you may need to restart the kernel to use updated packages.


In [2]:
import warnings; warnings.simplefilter('ignore')
import os, sys, subprocess, gc, re, math, time, queue, threading, contextlib

In [3]:
_BOXED_RE = re.compile(r"\\boxed\{(-?\d+(?:\.\d+)?(?:/\d+)?)\}")
_JUDGMENT_RE = re.compile(r"\*{0,2}Judgment:\*{0,2}\s*\[?\s*(-?\d+(?:\.\d+)?(?:/\d+)?)\s*\]?", re.IGNORECASE)
_FINAL_CHANNEL_RE = re.compile(r"<\|channel\|>final<\|message\|>(.*?)(?:<\|end\|>|$)", re.DOTALL)

GEN_SELECT_PROMPT = """\
You are judging {num_solutions} candidate solutions (numbered 0 through {max_idx}) to a math problem.

Your ONLY job is to pick the best answer FROM the solutions provided. You are a comparator, not a solver.

Problem:
{problem}

Solutions:
{solutions}

Instructions:

STRICT RULE — DO NOT RE-SOLVE: Do not attempt your own derivation, computation, or independent reasoning about the problem. Judges who re-derive answers get them wrong more often than the majority of solvers. If you catch yourself computing an answer or reasoning about the math independently, STOP and return to comparing the provided solutions.

STRICT RULE — ANSWER MUST COME FROM SOLUTIONS: Your final answer MUST be one of the answers that appears in the solution headers above. You may NOT output any answer that no solution proposed. There are exactly {num_solutions} solutions (numbered 0 through {max_idx}) — do not reference solutions outside this range.

Follow these steps:

1. Read each solution's final answer from the "(final answer: X)" label in its header. Group solutions by answer and count how many support each one. State the vote counts explicitly.

2. The answer with the most votes is the presumptive winner. In case of an exact tie, the answer from the lowest-indexed solution wins.

3. To override the majority, you must identify a specific, concrete error that you can QUOTE directly from every single majority solution's text — a wrong algebraic step, a wrong formula, a missed constraint, or a verifiable arithmetic mistake. The error must be visible in the solution text itself; your own alternative reasoning or re-derivation does NOT count as evidence. An error in an illustrative example does not invalidate the core derivation. Vague doubts do not count. If you cannot quote a concrete error from every majority solution, you MUST select the majority answer.

4. When a solution includes executed code that produced a result, trust the code output over manual reasoning, as long as the code correctly implements the problem constraints.

Your final answer must be a single integer that at least one solution proposed.

End with exactly:

\\boxed{{ANSWER}}

where ANSWER is the integer you select (e.g. \\boxed{{42}})."""

def _format_solutions(results):
    parts = []
    for i, r in enumerate(results):
        if r.get('Last Answer Turn'):
            text = r['Last Answer Turn']
        else:
            response = r.get('Generation') or '(no response)'
            boxed_pos = response.rfind('\\boxed{')
            if boxed_pos >= 0:
                start = max(0, boxed_pos - 2500)
                text = ('...' if start > 0 else '') + response[start:]
                if len(text) > 4000: text = '...' + text[-4000:]
            elif len(response) > 3000: text = '...' + response[-3000:]
            else: text = response
        raw_answer = r.get('Answer')
        if raw_answer is not None: answer = str(raw_answer)
        else:
            boxed_matches = _BOXED_RE.findall(r.get('Generation') or '')
            answer = boxed_matches[-1].strip() if boxed_matches else '(no answer)'
        parts.append(f'--- Solution {i} (final answer: {answer}) ---\n{text}')
    return '\n\n'.join(parts)

def _parse_judgment(text, raw_text=None):
    if raw_text:
        final_match = _FINAL_CHANNEL_RE.search(raw_text)
        if final_match:
            final_content = final_match.group(1)
            for regex in [_BOXED_RE, _JUDGMENT_RE]:
                matches = regex.findall(final_content)
                if matches: return matches[-1].strip()
    for regex in [_BOXED_RE, _JUDGMENT_RE]:
        for search_text in filter(None, [text, raw_text]):
            matches = regex.findall(search_text)
            if matches: return matches[-1].strip()
    return None

In [4]:
def set_env(archive, tmp):
    if not os.path.exists(tmp):
        os.makedirs(tmp, exist_ok=True)
        subprocess.run(['tar', '-xzf', archive, '-C', tmp], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index', '--find-links', f'{tmp}/wheels', 
                    'unsloth', 'trl', 'vllm', 'openai_harmony'], check=True)

set_env('/kaggle/input/notebooks/andreasbis/aimo-3-utils/wheels.tar.gz', '/kaggle/tmp/setup')

Looking in links: /kaggle/tmp/setup/wheels
Processing /kaggle/tmp/setup/wheels/unsloth-2025.12.9-py3-none-any.whl
Processing /kaggle/tmp/setup/wheels/trl-0.24.0-py3-none-any.whl
Processing /kaggle/tmp/setup/wheels/vllm-0.11.2-cp38-abi3-manylinux1_x86_64.whl
Processing /kaggle/tmp/setup/wheels/openai_harmony-0.0.8-cp38-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
Processing /kaggle/tmp/setup/wheels/unsloth_zoo-2025.12.7-py3-none-any.whl (from unsloth)
Processing /kaggle/tmp/setup/wheels/tyro-1.0.3-py3-none-any.whl (from unsloth)
Processing /kaggle/tmp/setup/wheels/xformers-0.0.33.post1-cp39-abi3-manylinux_2_28_x86_64.whl (from unsloth)
Processing /kaggle/tmp/setup/wheels/bitsandbytes-0.49.0-py3-none-manylinux_2_24_x86_64.whl (from unsloth)
Processing /kaggle/tmp/setup/wheels/datasets-4.3.0-py3-none-any.whl (from unsloth)
Processing /kaggle/tmp/setup/wheels/transformers-4.57.3-py3-none-any.whl (from unsloth)
Processing /kaggle/tmp/setup/wheels/prometheus_fastapi_instrumentator-7.1

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pyldavis 3.4.1 requires scikit-learn>=1.0.0, which is not installed.
ydata-profiling 4.18.1 requires matplotlib<=3.10,>=3.5, which is not installed.
librosa 0.11.0 requires scikit-learn>=1.1.0, which is not installed.
bigframes 2.31.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
bigframes 2.31.0 requires matplotlib>=3.7.1, which is not installed.
shap 0.50.0 requires scikit-learn, which is not installed.
pynndescent 0.6.0 requires scikit-learn>=0.18, which is not installed.
umap-learn 0.5.11 requires scikit-learn>=1.6, which is not installed.
sentence-transformers 5.2.0 requires scikit-learn, which is not installed.
google-adk 1.21.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
arviz 0.22.0 requires matplotlib>=3.8, which is not installed.
cuml-cu12 25.10

In [5]:
subprocess.run(['ls', '/kaggle/tmp/setup/tiktoken_encodings'])

cl100k_base.tiktoken
o200k_base.tiktoken


CompletedProcess(args=['ls', '/kaggle/tmp/setup/tiktoken_encodings'], returncode=0)

In [6]:
for k, v in [('TRANSFORMERS_NO_TF', '1'), ('TRANSFORMERS_NO_FLAX', '1'), ('CUDA_VISIBLE_DEVICES', '0'),
             ('TOKENIZERS_PARALLELISM', 'false'), ('TRITON_PTXAS_PATH', '/usr/local/cuda/bin/ptxas'),
             ('TIKTOKEN_ENCODINGS_BASE', '/kaggle/tmp/setup/tiktoken_encodings')]:
    os.environ[k] = v

In [7]:
from typing import Optional
from jupyter_client import KernelManager
from collections import Counter, defaultdict
from concurrent.futures import as_completed, ThreadPoolExecutor
import pandas as pd, polars as pl
from openai import OpenAI
from openai_harmony import (HarmonyEncodingName, load_harmony_encoding, SystemContent, ReasoningEffort, 
                             ToolNamespaceConfig, Author, Message, Role, TextContent, Conversation)
from transformers import set_seed
import kaggle_evaluation.aimo_3_inference_server

In [8]:
class CFG:
    system_prompt = ('You are a world-class International Mathematical Olympiad (IMO) competitor. '
                    'The final answer must be a non-negative integer between 0 and 99999. '
                    'You must place the final integer answer inside \\boxed{}.')
    tool_prompt = ('Use this tool to execute Python code. The environment is a stateful Jupyter notebook. '
                  'You must use print() to output results.')
    preference_prompt = 'You have access to `math`, `numpy` and `sympy` to solve the problem.'
    served_model_name, model_path = 'gpt-oss', '/kaggle/input/models/danielhanchen/gpt-oss-120b/transformers/default/1'
    kv_cache_dtype, dtype = 'fp8_e4m3', 'auto'
    high_problem_timeout, base_problem_timeout = 900, 300
    notebook_limit, server_timeout = 17400, 180
    session_timeout, jupyter_timeout, sandbox_timeout = 960, 6, 3
    stream_interval, context_tokens, buffer_tokens, search_tokens = 200, 65536, 512, 32
    top_logprobs, batch_size, early_stop, attempts, workers, turns = 5, 256, 4, 8, 20, 128
    gpu_memory_utilization, temperature, min_p, seed = 0.96, 1.0, 0.02, 42
    gen_select_threshold, judge_temperature, judge_max_tokens = 4, 0.0, 32768

In [9]:
set_seed(CFG.seed)

In [10]:
class AIMO3Template:
    def get_system_content(self, prompt, tool_cfg):
        return SystemContent.new().with_model_identity(prompt).with_reasoning_effort(
            reasoning_effort=ReasoningEffort.HIGH).with_tools(tool_cfg)
    
    def apply_chat_template(self, sys_prompt, usr_prompt, tool_cfg):
        return [Message.from_role_and_content(Role.SYSTEM, self.get_system_content(sys_prompt, tool_cfg)),
                Message.from_role_and_content(Role.USER, usr_prompt)]

In [11]:
class AIMO3Sandbox:
    _port_lock, _next_port = threading.Lock(), 50000
    
    @classmethod
    def _get_next_ports(cls, count=5):
        with cls._port_lock:
            ports = list(range(cls._next_port, cls._next_port + count))
            cls._next_port += count
            return ports
    
    def __init__(self, timeout):
        self._default_timeout, self._owns_kernel, self._client, self._km = timeout, False, None, None
        ports = self._get_next_ports(5)
        env = os.environ.copy()
        env.update({'PYDEVD_DISABLE_FILE_VALIDATION': '1', 'PYDEVD_WARN_EVALUATION_TIMEOUT': '0',
                   'JUPYTER_PLATFORM_DIRS': '1', 'PYTHONWARNINGS': 'ignore', 'MPLBACKEND': 'Agg'})
        self._km = KernelManager()
        self._km.shell_port, self._km.iopub_port, self._km.stdin_port, self._km.hb_port, self._km.control_port = ports
        self._km.start_kernel(env=env, extra_arguments=['--Application.log_level=CRITICAL'])
        self._client = self._km.blocking_client()
        self._client.start_channels()
        self._client.wait_for_ready(timeout=self._default_timeout)
        self._owns_kernel = True
        self.execute('import math, numpy, sympy, mpmath, itertools, collections\nmpmath.mp.dps = 64\n')
    
    def _format_error(self, tb):
        return ''.join(re.sub(r'\x1b\[[0-9;]*m', '', f) for f in tb 
                      if 'File "' not in f or 'ipython-input' in f)
    
    def execute(self, code, timeout=None):
        effective_timeout = timeout or self._default_timeout
        msg_id = self._client.execute(code, store_history=True, allow_stdin=False, stop_on_error=False)
        stdout, stderr, start = [], [], time.time()
        while True:
            if time.time() - start > effective_timeout:
                self._km.interrupt_kernel()
                return f'[ERROR] Execution timed out after {effective_timeout} seconds'
            try:
                msg = self._client.get_iopub_msg(timeout=1.0)
            except queue.Empty:
                continue
            if msg.get('parent_header', {}).get('msg_id') != msg_id: continue
            mt, c = msg.get('msg_type'), msg.get('content', {})
            if mt == 'stream':
                (stdout if c.get('name') == 'stdout' else stderr).append(c.get('text', ''))
            elif mt == 'error':
                stderr.append(self._format_error(c.get('traceback', [])))
            elif mt in {'execute_result', 'display_data'}:
                if txt := c.get('data', {}).get('text/plain'):
                    stdout.append(txt if txt.endswith('\n') else f'{txt}\n')
            elif mt == 'status' and c.get('execution_state') == 'idle':
                break
        out, err = ''.join(stdout), ''.join(stderr)
        return f'{out.rstrip()}\n{err}' if err and out else (err or out or '[WARN] No output. Use print() to see results.')
    
    def close(self):
        with contextlib.suppress(Exception):
            if self._client: self._client.stop_channels()
        if self._owns_kernel and self._km:
            with contextlib.suppress(Exception): self._km.shutdown_kernel(now=True)
            with contextlib.suppress(Exception): self._km.cleanup_resources()
    
    def reset(self):
        self.execute('%reset -f\nimport math, numpy, sympy, mpmath, itertools, collections\nmpmath.mp.dps = 64\n')
    
    def __del__(self):
        self.close()

In [12]:
class AIMO3Tool:
    def __init__(self, timeout, prompt, sandbox=None):
        self._local_jupyter_timeout, self._tool_prompt, self._jupyter_session = timeout, prompt, sandbox
        self._owns_session, self._execution_lock, self._init_lock = sandbox is None, threading.Lock(), threading.Lock()
    
    def _ensure_session(self):
        if self._jupyter_session is None:
            with self._init_lock:
                if self._jupyter_session is None:
                    self._jupyter_session = AIMO3Sandbox(timeout=self._local_jupyter_timeout)
    
    def _ensure_last_print(self, code):
        lines = code.strip().split('\n')
        if not lines: return code
        last = lines[-1].strip()
        if any(x in last for x in ['print', 'import']) or not last or last.startswith('#'): return code
        lines[-1] = 'print(' + last + ')'
        return '\n'.join(lines)
    
    @property
    def instruction(self): return self._tool_prompt
    
    @property
    def tool_config(self): return ToolNamespaceConfig(name='python', description=self.instruction, tools=[])
    
    def _make_response(self, output, channel=None):
        msg = Message(author=Author(role=Role.TOOL, name='python'), 
                     content=[TextContent(text=output)]).with_recipient('assistant')
        return msg.with_channel(channel) if channel else msg
    
    def process_sync_plus(self, message):
        self._ensure_session()
        final_script = self._ensure_last_print(message.content[0].text)
        with self._execution_lock:
            try:
                output = self._jupyter_session.execute(final_script)
            except TimeoutError as exc:
                output = f'[ERROR] {exc}'
        return [self._make_response(output, channel=message.channel)]

In [13]:
class AIMO3Solver:
    def __init__(self, cfg, port=8000):
        self.cfg, self.port = cfg, port
        self.base_url, self.api_key = f'http://0.0.0.0:{port}/v1', 'sk-local'
        self.template, self.encoding = AIMO3Template(), load_harmony_encoding(HarmonyEncodingName.HARMONY_GPT_OSS)
        self.stop_token_ids = self.encoding.stop_tokens_for_assistant_actions()
        self._preload_model_weights()
        self.server_process = self._start_server()
        self.client = OpenAI(base_url=self.base_url, api_key=self.api_key, timeout=self.cfg.session_timeout)
        self._wait_for_server()
        self._initialize_kernels()
        self.notebook_start_time, self.problems_remaining = time.time(), 50
    
    def _preload_model_weights(self):
        print(f'Loading model weights from {self.cfg.model_path} into OS Page Cache...')
        start, files, total = time.time(), [], 0
        for root, _, fnames in os.walk(self.cfg.model_path):
            for fn in fnames:
                fp = os.path.join(root, fn)
                if os.path.isfile(fp):
                    files.append(fp)
                    total += os.path.getsize(fp)
        with ThreadPoolExecutor(max_workers=self.cfg.workers) as ex:
            list(ex.map(lambda p: open(p, 'rb').read(), files))
        print(f'Processed {len(files)} files ({total/1e9:.2f} GB) in {time.time()-start:.2f} seconds.\n')
    
    def _start_server(self):
        cmd = [sys.executable, '-m', 'vllm.entrypoints.openai.api_server', '--seed', str(self.cfg.seed),
               '--model', self.cfg.model_path, '--served-model-name', self.cfg.served_model_name,
               '--tensor-parallel-size', '1', '--max-num-seqs', str(self.cfg.batch_size),
               '--gpu-memory-utilization', str(self.cfg.gpu_memory_utilization), '--host', '0.0.0.0',
               '--port', str(self.port), '--dtype', self.cfg.dtype, '--kv-cache-dtype', self.cfg.kv_cache_dtype,
               '--max-model-len', str(self.cfg.context_tokens), '--stream-interval', str(self.cfg.stream_interval),
               '--async-scheduling', '--disable-log-stats', '--enable-prefix-caching']
        self.log_file = open('vllm_server.log', 'w')
        return subprocess.Popen(cmd, stdout=self.log_file, stderr=subprocess.STDOUT, start_new_session=True)
    
    def _wait_for_server(self):
        print('Waiting for vLLM server...')
        start = time.time()
        for _ in range(self.cfg.server_timeout):
            if (rc := self.server_process.poll()) is not None:
                self.log_file.flush()
                raise RuntimeError(f'Server died with code {rc}. Full logs:\n{open("vllm_server.log").read()}\n')
            try:
                self.client.models.list()
                print(f'Server is ready (took {time.time()-start:.2f} seconds).\n')
                return
            except Exception:
                time.sleep(1)
        raise RuntimeError('Server failed to start (timeout).\n')
    
    def _initialize_kernels(self):
        print(f'Initializing {self.cfg.workers} persistent Jupyter kernels...')
        start = time.time()
        self.sandbox_pool = queue.Queue()
        with ThreadPoolExecutor(max_workers=self.cfg.workers) as ex:
            for future in as_completed([ex.submit(lambda: AIMO3Sandbox(timeout=self.cfg.jupyter_timeout)) 
                                       for _ in range(self.cfg.workers)]):
                self.sandbox_pool.put(future.result())
        print(f'Kernels initialized in {time.time()-start:.2f} seconds.\n')
    
    def _scan_for_answer(self, text):
        for pattern in [r'\\boxed\s*\{\s*([0-9,]+)\s*\}', r'final\s+answer\s+is\s*([0-9,]+)']:
            if matches := re.findall(pattern, text, re.IGNORECASE):
                try:
                    val = int(matches[-1].replace(',', ''))
                    if 0 <= val <= 99999: return val
                except ValueError: pass
        return None
    
    def _compute_mean_entropy(self, logprobs):
        if not logprobs: return float('inf')
        total, count = 0.0, 0
        for top_lp in logprobs:
            if isinstance(top_lp, dict) and top_lp:
                ent = sum(-math.exp(lp)*math.log2(math.exp(lp)) for lp in top_lp.values() if math.exp(lp) > 0)
                total += ent
                count += 1
        return total/count if count else float('inf')
    
    def _process_attempt(self, problem, sys_prompt, idx, stop_evt, deadline):
        if stop_evt.is_set() or time.time() > deadline:
            return {'Attempt': idx+1, 'Answer': None, 'Python Calls': 0, 'Python Errors': 0, 
                   'Response Length': 0, 'Entropy': float('inf'), 'Generation': '', 'Last Answer Turn': None}
        local_tool, sandbox, py_calls, py_errs, total_toks, ans, logprobs = None, None, 0, 0, 0, None, []
        generation_chunks, last_answer_turn = [], None
        seed = int(math.pow(self.cfg.seed + idx, 2))
        try:
            sandbox = self.sandbox_pool.get(timeout=self.cfg.sandbox_timeout)
            local_tool = AIMO3Tool(self.cfg.jupyter_timeout, self.cfg.tool_prompt, sandbox)
            conv = Conversation.from_messages(self.template.apply_chat_template(
                sys_prompt, problem, local_tool.tool_config))
            for _ in range(self.cfg.turns):
                if stop_evt.is_set() or time.time() > deadline: break
                prompt_ids = self.encoding.render_conversation_for_completion(conv, Role.ASSISTANT)
                if (max_toks := self.cfg.context_tokens - len(prompt_ids)) < self.cfg.buffer_tokens: break
                stream = self.client.completions.create(model=self.cfg.served_model_name, 
                    temperature=self.cfg.temperature, logprobs=self.cfg.top_logprobs, max_tokens=max_toks,
                    prompt=prompt_ids, seed=seed, stream=True, extra_body={
                        'min_p': self.cfg.min_p, 'stop_token_ids': self.stop_token_ids, 'return_token_ids': True})
                try:
                    tok_buf, txt_chunks = [], []
                    for chunk in stream:
                        if stop_evt.is_set() or time.time() > deadline: break
                        if new_toks := chunk.choices[0].token_ids:
                            tok_buf.extend(new_toks)
                            total_toks += len(new_toks)
                            txt_chunks.append(chunk.choices[0].text)
                            generation_chunks.append(chunk.choices[0].text)
                            if (clp := chunk.choices[0].logprobs) and clp.top_logprobs:
                                logprobs.extend(clp.top_logprobs)
                        if '}' in chunk.choices[0].text and (ans := self._scan_for_answer(
                            ''.join(txt_chunks[-self.cfg.search_tokens:]))):
                            last_answer_turn = ''.join(txt_chunks)
                            break
                finally:
                    stream.close()
                if ans or not tok_buf: break
                new_msgs = self.encoding.parse_messages_from_completion_tokens(tok_buf, Role.ASSISTANT)
                conv.messages.extend(new_msgs)
                last = new_msgs[-1]
                if last.channel == 'final':
                    ans = self._scan_for_answer(last.content[0].text)
                    last_answer_turn = last.content[0].text
                    break
                if last.recipient == 'python':
                    py_calls += 1
                    resp = local_tool.process_sync_plus(last)
                    if any(x in (txt := resp[0].content[0].text) for x in ['[ERROR]', 'Traceback', 'Error:']):
                        py_errs += 1
                    conv.messages.extend(resp)
        except Exception: py_errs += 1
        finally:
            if sandbox:
                sandbox.reset()
                self.sandbox_pool.put(sandbox)
        return {'Attempt': idx+1, 'Response Length': total_toks, 'Python Calls': py_calls, 
               'Python Errors': py_errs, 'Entropy': self._compute_mean_entropy(logprobs), 'Answer': ans,
               'Generation': ''.join(generation_chunks), 'Last Answer Turn': last_answer_turn}
    
    def _judge_solutions(self, problem, detailed_results, deadline=0):
        valid_results = [r for r in detailed_results if r.get('Generation')]
        if len(valid_results) < 2:
            print('[Gen-select] Fewer than 2 valid generations, skipping judge')
            return None
        num_solutions = len(valid_results)
        prompt = GEN_SELECT_PROMPT.format(num_solutions=num_solutions, max_idx=num_solutions-1,
                                          problem=problem, solutions=_format_solutions(valid_results))
        timeout = max(10, deadline - time.time()) if deadline else self.cfg.session_timeout
        t0 = time.time()
        try:
            resp = self.client.chat.completions.create(model=self.cfg.served_model_name,
                messages=[{'role': 'user', 'content': prompt}], temperature=self.cfg.judge_temperature,
                max_tokens=self.cfg.judge_max_tokens, timeout=timeout)
            text = resp.choices[0].message.content or ''
            elapsed = time.time() - t0
            selected = _parse_judgment(text, raw_text=text)
            print(f'[Gen-select] Judge responded in {elapsed:.2f}s, selected: {selected}')
            if selected is not None:
                try: selected_int = int(selected)
                except (ValueError, TypeError):
                    print(f'[Gen-select] Judge answer {selected!r} not an integer, falling back')
                    return None
                proposed = {r.get('Answer') for r in valid_results if r.get('Answer') is not None}
                if selected_int in proposed:
                    print(f'[Gen-select] Judge selected answer: {selected_int}')
                    return selected_int
                else:
                    print(f'[Gen-select] Judge answer {selected_int} not in proposed {proposed}, falling back')
                    return None
            else:
                print('[Gen-select] Could not parse judgment answer, falling back')
                return None
        except Exception as e:
            print(f'[Gen-select] Judge call failed after {time.time()-t0:.2f}s: {e}')
            return None
    
    def _select_answer(self, results, problem='', deadline=0):
        ans_weights, ans_votes = defaultdict(float), defaultdict(int)
        for r in results:
            if (a := r['Answer']) is not None:
                ans_weights[a] += 1.0/max(r['Entropy'], 1e-9)
                ans_votes[a] += 1
        scored = sorted([{'answer': a, 'votes': ans_votes[a], 'score': w} 
                        for a, w in ans_weights.items()], key=lambda x: x['score'], reverse=True)
        display(pd.DataFrame([(s['answer'], s['votes'], s['score']) for s in scored], 
                            columns=['Answer', 'Votes', 'Score']).round({'Score': 3}))
        final = scored[0]['answer'] if scored else 0
        majority_votes = scored[0]['votes'] if scored else 0
        if majority_votes < self.cfg.gen_select_threshold and problem:
            print(f'[Gen-select] Majority votes ({majority_votes}) < threshold ({self.cfg.gen_select_threshold}), invoking judge')
            judge_answer = self._judge_solutions(problem, results, deadline=deadline)
            if judge_answer is not None:
                print(f'[Gen-select] Using judge answer: {judge_answer}')
                final = judge_answer
            else:
                print(f'[Gen-select] Judge failed, falling back to weighted-entropy winner: {final}')
        print(f'\nFinal Answer: {final}\n')
        return final
    
    def solve_problem(self, problem):
        print(f'\nProblem: {problem}\n')
        user_input = f'{problem} {self.cfg.preference_prompt}'
        time_left = self.cfg.notebook_limit - (time.time() - self.notebook_start_time)
        budget = max(self.cfg.base_problem_timeout, 
                    min(time_left - max(0, self.problems_remaining-1)*self.cfg.base_problem_timeout, 
                        self.cfg.high_problem_timeout))
        deadline = time.time() + budget
        print(f'Budget: {budget:.2f} seconds | Deadline: {deadline:.2f}\n')
        results, valid, stop_evt = [], [], threading.Event()
        with ThreadPoolExecutor(max_workers=self.cfg.workers) as ex:
            futures = [ex.submit(self._process_attempt, user_input, self.cfg.system_prompt, i, stop_evt, deadline)
                      for i in range(self.cfg.attempts)]
            for future in as_completed(futures):
                try:
                    if (r := future.result())['Answer'] is not None:
                        valid.append(r['Answer'])
                    results.append(r)
                    if (cnts := Counter(valid).most_common(1)) and cnts[0][1] >= self.cfg.early_stop:
                        stop_evt.set()
                        for f in futures: f.cancel()
                        break
                except Exception as exc:
                    print(f'Future failed: {exc}')
        self.problems_remaining = max(0, self.problems_remaining - 1)
        if results:
            df = pd.DataFrame(results)
            df['Entropy'] = df['Entropy'].round(3)
            df['Answer'] = df['Answer'].astype('Int64')
            display(df.drop(columns=['Generation', 'Last Answer Turn'], errors='ignore'))
        return self._select_answer(results, problem=user_input, deadline=deadline) if valid else 0
    
    def __del__(self):
        if hasattr(self, 'server_process'):
            self.server_process.terminate()
            self.server_process.wait()
        if hasattr(self, 'log_file'): self.log_file.close()
        if hasattr(self, 'sandbox_pool'):
            while not self.sandbox_pool.empty():
                with contextlib.suppress(Exception): self.sandbox_pool.get_nowait().close()

In [14]:
solver = AIMO3Solver(CFG)

Loading model weights from /kaggle/input/models/danielhanchen/gpt-oss-120b/transformers/default/1 into OS Page Cache...
Processed 26 files (65.28 GB) in 83.38 seconds.

Waiting for vLLM server...
Server is ready (took 121.40 seconds).

Initializing 20 persistent Jupyter kernels...
Kernels initialized in 3.30 seconds.



In [15]:
def predict(id_: pl.DataFrame, question: pl.DataFrame, answer: Optional[pl.DataFrame] = None) -> pl.DataFrame:
    gc.disable()
    final_answer = solver.solve_problem(question.item(0))
    gc.enable()
    gc.collect()
    return pl.DataFrame({'id': id_.item(0), 'answer': final_answer})

In [16]:
inference_server = kaggle_evaluation.aimo_3_inference_server.AIMO3InferenceServer(predict)
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    inference_server.serve()
else:
    inference_server.run_local_gateway(('/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/reference.csv',))


Problem: Define a function $f \colon \mathbb{Z}_{\geq 1} \to \mathbb{Z}_{\geq 1}$ by
\begin{equation*}
    f(n) = \sum_{i = 1}^n \sum_{j = 1}^n j^{1024} \left\lfloor\frac1j + \frac{n-i}{n}\right\rfloor.
\end{equation*}
Let $M=2 \cdot 3 \cdot 5 \cdot 7 \cdot 11 \cdot 13$ and let $N = f{\left(M^{15}\right)} - f{\left(M^{15}-1\right)}$. Let $k$ be the largest non-negative integer such that $2^k$ divides $N$. What is the remainder when $2^k$ is divided by $5^7$?

Budget: 900.00 seconds | Deadline: 1773118013.14



,Attempt,Response Length,Python Calls,Python Errors,Entropy,Answer
0,3,4170,1,0,0.663,32951
1,6,4495,4,0,0.617,32951
2,1,5637,5,0,0.611,32951
3,2,5892,2,0,0.647,32951


,Answer,Votes,Score
0,32951,4,6.311



Final Answer: 32951


Problem: Let $f \colon \mathbb{Z}_{\geq 1} \to \mathbb{Z}_{\geq 1}$ be a function such that for all positive integers $m$ and $n$, 
\begin{equation*}
    f(m) + f(n) = f(m + n + mn).
\end{equation*}
Across all functions $f$ such that $f(n) \leq 1000$ for all $n \leq 1000$, how many different values can $f(2024)$ take?

Budget: 900.00 seconds | Deadline: 1773118080.61



,Attempt,Response Length,Python Calls,Python Errors,Entropy,Answer
0,6,8467,6,0,0.845,580
1,3,10586,9,0,0.849,580
2,2,10250,7,2,0.754,580
3,7,12065,8,0,0.711,580


...

,Answer,Votes,Score
0,520,4,4.211



Final Answer: 520


Problem: A tournament is held with $2^{20}$ runners each of which has a different running speed. In each race, two runners compete against each other with the faster runner always winning the race. The competition consists of $20$ rounds with each runner starting with a score of $0$. In each round, the runners are paired in such a way that in each pair, both runners have the same score at the beginning of the round. The winner of each race in the $i^{\text{th}}$ round receives $2^{20-i}$ points and the loser gets no points.

At the end of the tournament, we rank the competitors according to their scores. Let $N$ denote the number of possible orderings of the competitors at the end of the tournament. Let $k$ be the largest positive integer such that $10^k$ divides $N$. What is the remainder when $k$ is divided by $10^{5}$?

Budget: 900.00 seconds | Deadline: 1773119829.41



,Attempt,Response Length,Python Calls,Python Errors,Entropy,Answer
0,6,9018,5,0,1.012,62140
1,5,10988,5,0,0.866,21818
2,3,14381,15,0,0.813,21818
3,7,15020,1,0,0.920,21818
4,1,16504,27,4,0.771,21818


,Answer,Votes,Score
0,21818,4,4.770
1,62140,1,0.988



Final Answer: 21818

